# 00 – EDA: FREUID 2026 Sample Dataset
Explore the 13-image sample: labels, document types, is_digital split, and image characteristics.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from src.utils.config import load_config

ROOT = Path('..')
cfg = load_config(ROOT / 'config.yaml')
IMG_ROOT = ROOT / cfg.data.root  # e.g. ROOT/train_sample for sample data
df = pd.read_csv(ROOT / 'train_sample_labels.csv')
print(df.dtypes)
df.head()

In [ ]:
print(f"Total samples : {len(df)}")
print(f"Genuine (0)   : {(df.label==0).sum()}")
print(f"Fraud   (1)   : {(df.label==1).sum()}")
print(f"Digital       : {df.is_digital.map({True:1,False:0,'True':1,'False':0}).sum()}")
print(f"Recaptured    : {(~df.is_digital.map({True:True,False:False,'True':True,'False':False})).sum()}")
print(f"\nDocument types:\n{df['type'].value_counts()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Label distribution
df['label'].value_counts().plot.bar(ax=axes[0], color=['steelblue','tomato'])
axes[0].set_title('Label distribution'); axes[0].set_xlabel(''); axes[0].set_xticklabels(['Genuine','Fraud'], rotation=0)

# is_digital
df['is_digital'].astype(str).value_counts().plot.bar(ax=axes[1], color=['mediumseagreen','orange'])
axes[1].set_title('is_digital'); axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Document type
df['type'].value_counts().plot.bar(ax=axes[2], color='mediumpurple')
axes[2].set_title('Document type'); axes[2].tick_params(axis='x', labelrotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tab: label vs is_digital
df['is_digital_bool'] = df['is_digital'].map({True:True, False:False, 'True':True, 'False':False})
pd.crosstab(df['label'], df['is_digital_bool'], margins=True,
            rownames=['label'], colnames=['is_digital'])

In [ ]:
# Visualise all 13 images with label and type annotations
n = len(df)
ncols = 5
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*3))
axes = axes.flatten()

for i, (_, row) in enumerate(df.iterrows()):
    img = Image.open(IMG_ROOT / row['image_path']).convert('RGB')
    axes[i].imshow(img)
    color = 'red' if row['label'] == 1 else 'green'
    dig = 'D' if str(row['is_digital']) in ('True','1') else 'R'
    axes[i].set_title(f"{row['type']}\nlabel={row['label']} [{dig}]", fontsize=8, color=color)
    for spine in axes[i].spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)
    axes[i].axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

red_patch = mpatches.Patch(color='red', label='Fraud (1)')
grn_patch = mpatches.Patch(color='green', label='Genuine (0)')
fig.legend(handles=[grn_patch, red_patch], loc='lower right', fontsize=10)
plt.suptitle('FREUID Sample Images  (D=digital, R=recaptured)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Image size distribution
sizes = []
for _, row in df.iterrows():
    img = Image.open(IMG_ROOT / row['image_path'])
    sizes.append({'id': row['id'], 'w': img.size[0], 'h': img.size[1],
                  'aspect': img.size[0]/img.size[1], 'label': row['label']})
sizes_df = pd.DataFrame(sizes)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(sizes_df['w'], bins=8, color='steelblue'); axes[0].set_title('Width')
axes[1].hist(sizes_df['h'], bins=8, color='coral');    axes[1].set_title('Height')
axes[2].hist(sizes_df['aspect'], bins=8, color='gold'); axes[2].set_title('Aspect ratio (w/h)')
plt.tight_layout(); plt.show()

print(sizes_df[['w','h','aspect']].describe().round(1))